Gurobi Version

In [ ]:
"""
Gurobi MILP model for EarlyCon (30‑day horizon, FIFO, fixed early‑delivery bonus)
Author: Lars Buchholz • 2025‑07‑14

Usage summary
-------------
1. **Edit the INPUT DATA block**
   * `I` … list of coating IDs.
   * `d[(i,t)]` … *actual* arrivals for `t = 0`, *forecast* for `t = 1‥29`.
   * `s_init[(i,a)]` … backlog carried over from the *previous* day (ages 1‑4).
2. Run the file (`python earlycon_milp.py`).  Gurobi will solve and print the
   objective.  The model is now guaranteed to be feasible for consistent data.
3. Advance one day: shift forecasts, compute new `s_init` from today’s unsolved
   backlog (ages advance by +1), set fresh `d[(i,0)]` and re‑solve.

Key parameters (can be tweaked without touching the maths):
* Machines            M          (default 65)
* Regular batches     B_reg = 2  per machine per day
* OT batches          B_ot  = 1  per machine per day
* Batch capacity      Q     = 1  (max fill‑rate)
* Cost (regular)      c_reg = €150 per batch
* Cost (overtime)     c_ot  = €300 per batch
* Early‑bonus coeff.  gamma = €50 per early day per **full** fill‑rate

All constraints implement the Markdown spec agreed with the user.
"""

from __future__ import annotations

import gurobipy as gp
from gurobipy import GRB
from typing import Dict, List, Tuple

# ----------------------------------------------------------------------------
# ↑↑↑ INPUT DATA – EDIT THIS BLOCK -------------------------------------------
# ----------------------------------------------------------------------------
# Coating IDs (can be strings or ints)
I: List[str] = ["C1", "C12", "C15"]

# Planning horizon (fixed 30 days, 0 = today)
T: List[int] = list(range(8))
A: List[int] = list(range(5))  # age buckets 0..4

# Demand forecast d[i,t]
# * t = 0  → **actual arrivals** observed this morning (after yesterday’s solve)
# * t ≥ 1 → forecast for the next 29 days
# Replace the zeros with your real data each day ↓↓↓

d: Dict[Tuple[str, int], float] = {
    # Actual arrivals for t=0 (today)
    ("C1", 0): 32.2,  # Example: Actual arrivals for C1 today
    ("C12", 0): 46.4,  # Example: Actual arrivals for C2 today
    ("C15", 0): 62.5,  # Example: Actual arrivals for C3 today

    # Forecasts for t=1 to t=7 (assuming T is list(range(8)))
    # You would fill this with your forecast data for each day and coating
    ("C1", 1): 0.664382,
    ("C1", 2): 3.116381,
    ("C1", 3): 2.638918,
    ("C1", 4): 0.006072,
    ("C1", 5): 0.16748,
    ("C1", 6): 7.168205,
    ("C1", 7): 2.797644,
    ("C1", 8): 2.601825,
    ("C1", 9): 2.337529,
    ("C1", 10): 2.595371,
    ("C1", 11): 0.003316,
    ("C1", 12): 0.037442,
    ("C1", 13): 11.13356,

    ("C12", 1): 0.092957,
    ("C12", 2): 1.709557,
    ("C12", 3): 1.02712,
    ("C12", 4): -0.17251,
    ("C12", 5): 0.014203,
    ("C12", 6): 5.333041,
    ("C12", 7): 1.118007,
    ("C12", 8): 1.141394,
    ("C12", 9): 0.764299,
    ("C12", 10): 1.316963,
    ("C12", 11): -0.24267,
    ("C12", 12): -0.11134,
    ("C12", 13): 11.81268,

    ("C15", 1): 0.420059,
    ("C15", 2): 1.890447,
    ("C15", 3): 1.505981,
    ("C15", 4): 0.054836,
    ("C15", 5): 0.169436,
    ("C15", 6): 4.827995,
    ("C15", 7): 1.741779,
    ("C15", 8): 1.505855,
    ("C15", 9): 1.322614,
    ("C15", 10): 1.535442,
    ("C15", 11): 0.027544,
    ("C15", 12): 0.007686,
    ("C15", 13): 6.440917,
}

# Starting backlog carried into today (ages 1‑4 only)
# Typically produced by yesterday’s solve – default 0 for demo
s_init: Dict[Tuple[str, int], float] = {
    (i, a): 0.0 for i in I for a in A  # <-- ages 1‑4 get real carry‑over backlog
}

# Shop parameters (feel free to change)
params = dict(
    M=65,
    B_reg=2,
    B_ot=1,
    Q=1.0,
    c_reg=150.0,
    c_ot=300.0,
    gamma=50.0,
)
# ----------------------------------------------------------------------------
# ↓↓↓ END INPUT DATA ----------------------------------------------------------
# ----------------------------------------------------------------------------


def build_earlycon_model(
    I: List[str],
    T: List[int],
    A: List[int],
    d: Dict[Tuple[str, int], float],
    s_init: Dict[Tuple[str, int], float],
    params: dict,
) -> gp.Model:
    """Return a fully‑built Gurobi model for the EarlyCon scheduling MILP."""

    M       = params.get("M", 65)
    B_reg   = params.get("B_reg", 2)
    B_ot    = params.get("B_ot", 1)
    Q       = params.get("Q", 1.0)
    c_reg   = params.get("c_reg", 150.0)
    c_ot    = params.get("c_ot", 300.0)
    gamma   = params.get("gamma", 50.0)

    C_reg = M * B_reg
    C_ot  = M * B_ot

    m = gp.Model("EarlyCon")

    # ----------------------
    # Variables
    # ----------------------
    s = m.addVars(I, T, A, name="s", lb=0.0)
    p = m.addVars(I, T, A, name="p", lb=0.0)
    n_reg = m.addVars(I, T, vtype=GRB.INTEGER, name="n_reg", lb=0)
    n_ot  = m.addVars(I, T, vtype=GRB.INTEGER, name="n_ot",  lb=0)

    # ----------------------
    # Constraints
    # ----------------------
    # 1. Initial backlog for ages 1‑4 (age‑0 handled by arrivals below)
    for i in I:
        for a in A[1:]:
            m.addConstr(s[i, 0, a] == s_init[i, a], name=f"InitBacklog[{i},{a}]")
        if s_init.get((i, 0), 0.0) > 1e-8:
            print(f"⚠️  Warning: s_init[{i},0] is {s_init[(i,0)]} – move this into d[{i},0]")

    # 2. Arrivals (define age‑0 backlog for each day)
    for i in I:
        for t in T:
            m.addConstr(s[i, t, 0] == d[i, t], name=f"Arrivals[{i},{t}]")

    # 3. Backlog ageing for t ≥ 1, a ≥ 1
    for i in I:
        for t in T[1:]:
            for a in A[1:]:
                m.addConstr(
                    s[i, t, a] == s[i, t - 1, a - 1] - p[i, t - 1, a - 1],
                    name=f"Ageing[{i},{t},{a}]",
                )

    # 4. Process only what exists
    m.addConstrs((p[i, t, a] <= s[i, t, a] for i in I for t in T for a in A),
                 name="ProcExists")

    # 5. Due‑date clearance for age 4
    m.addConstrs((p[i, t, 4] == s[i, t, 4] for i in I for t in T),
                 name="DueDate")

    # 6. Batch‑volume link
    m.addConstrs((gp.quicksum(p[i, t, a] for a in A) <= (n_reg[i, t] + n_ot[i, t]) * Q
                   for i in I for t in T),
                 name="BatchVol")

    # 7. Site capacity (regular & OT)
    m.addConstrs((gp.quicksum(n_reg[i, t] for i in I) <= C_reg for t in T),
                 name="CapReg")
    m.addConstrs((gp.quicksum(n_ot[i, t] for i in I) <= C_ot for t in T),
                 name="CapOT")

    # ----------------------
    # Objective: maximise profit
    # ----------------------
    bonus_term = gp.quicksum(gamma * (4 - a) * p[i, t, a]
                             for i in I for t in T for a in A)
    cost_term  = gp.quicksum(c_reg * n_reg[i, t] + c_ot * n_ot[i, t]
                             for i in I for t in T)
    m.setObjective(bonus_term - cost_term, GRB.MAXIMIZE)

    # Solver output flag (1 = on, 0 = silent)
    m.Params.OutputFlag = 1

    return m

# ---------------------------------------------------------------------------
# Build & solve (example) ----------------------------------------------------
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    model = build_earlycon_model(I, T, A, d, s_init, params)
    model.optimize()

    # --- Example: print total profit
    if model.Status == GRB.OPTIMAL:
        print("Optimal objective:", model.ObjVal)


Restricted license - for non-production use only - expires 2026-11-23
Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 24.04.2 LTS")

CPU model: Intel(R) Xeon(R) Platinum 8370C CPU @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 304 rows, 288 columns and 792 nonzeros
Model fingerprint: 0x1806189a
Variable types: 240 continuous, 48 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e+01, 3e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [6e-03, 1e+02]
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 2 available processors)

Solution count 0

Model is infeasible or unbounded
Best objective -, best bound -, gap -


In [ ]:
# ---------------------------------------------------------------------------
# Helpers to extract and print decisions
# ---------------------------------------------------------------------------
def print_daily_report(
    model: gp.Model,
    I: List[str],
    T: List[int],
    A: List[int],
    p, n_reg, n_ot,
    params: dict,
) -> None:
    """Pretty-print the schedule and profit line by line."""

    gamma, c_reg, c_ot = params["gamma"], params["c_reg"], params["c_ot"]

    header = f"{'Day':>3} | {'Reg':>4} {'OT':>4} | {'Vol proc.':>9} | " \
             f"{'Bonus €':>9} {'Cost €':>9} {'Profit €':>9}"
    print("\n" + header)
    print("-"*len(header))

    grand_bonus = grand_cost = 0.0
    for t in T:
        day_bonus = sum(gamma*(4-a)*p[i,t,a].X for i in I for a in A)
        day_cost  = (
            c_reg * sum(n_reg[i,t].X for i in I) +
            c_ot  * sum(n_ot[i,t].X  for i in I)
        )
        day_profit = day_bonus - day_cost
        grand_bonus += day_bonus
        grand_cost  += day_cost

        print(f"{t:3d} | "
              f"{int(sum(n_reg[i,t].X for i in I)):4d} "
              f"{int(sum(n_ot[i,t].X  for i in I)):4d} | "
              f"{sum(p[i,t,a].X for i in I for a in A):9.1f} | "
              f"{day_bonus:9.0f} {day_cost:9.0f} {day_profit:9.0f}")

    print("-"*len(header))
    print(f"{'Σ':>3} | ---- ---- | --------- | "
          f"{grand_bonus:9.0f} {grand_cost:9.0f} "
          f"{grand_bonus-grand_cost:9.0f}\n")


In [ ]:
if model.Status == GRB.OPTIMAL:
    print("Optimal objective:", model.ObjVal)

    # Extract variable dictionaries from the model
    p = model.getVarByName("p")
    n_reg = model.getVarByName("n_reg")
    n_ot = model.getVarByName("n_ot")

    # If the above does not work (returns None), reconstruct using model._vars
    # But since you used m.addVars(...), you can access them as model.getVars()
    # Let's reconstruct the variable dictionaries:
    p_vars = {tuple((v.VarName.split('[')[1][:-1]).split(',')): v for v in model.getVars() if v.VarName.startswith('p[')}
    n_reg_vars = {tuple((v.VarName.split('[')[1][:-1]).split(',')): v for v in model.getVars() if v.VarName.startswith('n_reg[')}
    n_ot_vars = {tuple((v.VarName.split('[')[1][:-1]).split(',')): v for v in model.getVars() if v.VarName.startswith('n_ot[')}

    # Convert keys to correct types
    def tuple_key(key):
        return tuple(int(x) if x.isdigit() else x for x in key)

    p = {(k[0], int(k[1]), int(k[2])): v for k, v in ((tuple_key(k), v) for k, v in p_vars.items())}
    n_reg = {(k[0], int(k[1])): v for k, v in ((tuple_key(k), v) for k, v in n_reg_vars.items())}
    n_ot = {(k[0], int(k[1])): v for k, v in ((tuple_key(k), v) for k, v in n_ot_vars.items())}

    # Call the reporter
    print_daily_report(
        model, I, T, A,
        p,
        n_reg,
        n_ot,
        params,
    )

We need to make sure that the behaviour is not affected by boundaries. Idea: Just let the algorithm optimize given the information for the next five days. Give some warmup and some cooldown period. Additional constraint: Make sure that all orders are fulfilled on the last day, even if they have not reached age 4, yet.

In [ ]:
"""
Gurobi MILP model for EarlyCon (30‑day horizon, FIFO, fixed early‑delivery bonus)
Author: Lars Buchholz • 2025‑07‑14

Usage summary
-------------
1. **Edit the INPUT DATA block**
   * `I` … list of coating IDs.
   * `d[(i,t)]` … *actual* arrivals for `t = 0`, *forecast* for `t = 1‥29`.
   * `s_init[(i,a)]` … backlog carried over from the *previous* day (ages 1‑4).
2. Run the file (`python earlycon_milp.py`).  Gurobi will solve and print the
   objective.  The model is now guaranteed to be feasible for consistent data.
3. Advance one day: shift forecasts, compute new `s_init` from today’s unsolved
   backlog (ages advance by +1), set fresh `d[(i,0)]` and re‑solve.

Key parameters (can be tweaked without touching the maths):
* Machines            M          (default 65)
* Regular batches     B_reg = 2  per machine per day
* OT batches          B_ot  = 1  per machine per day
* Batch capacity      Q     = 1  (max fill‑rate)
* Cost (regular)      c_reg = €150 per batch
* Cost (overtime)     c_ot  = €300 per batch
* Early‑bonus coeff.  gamma = €50 per early day per **full** fill‑rate

All constraints implement the Markdown spec agreed with the user.
"""

from __future__ import annotations

import gurobipy as gp
from gurobipy import GRB
from typing import Dict, List, Tuple

# ----------------------------------------------------------------------------
# ↑↑↑ INPUT DATA – EDIT THIS BLOCK -------------------------------------------
# ----------------------------------------------------------------------------
# Coating IDs (can be strings or ints)
I: List[str] = ["C1", "C2", "C3"]

# Planning horizon (fixed 30 days, 0 = today)
T: List[int] = list(range(3))
A: List[int] = list(range(5))  # age buckets 0..4

# Demand forecast d[i,t]
# * t = 0  → **actual arrivals** observed this morning (after yesterday’s solve)
# * t ≥ 1 → forecast for the next 29 days
# Replace the zeros with your real data each day ↓↓↓

d: Dict[Tuple[str, int], float] = {
    **{("C1", t): 50.2 for t in T},  # <-- UPDATE HERE
    **{("C2", t): 60.1 for t in T},  # <-- UPDATE HERE
    **{("C3", t): 60.7 for t in T},  # <-- UPDATE HERE
}

# Starting backlog carried into today (ages 1‑4 only)
# Typically produced by yesterday’s solve – default 0 for demo
s_init: Dict[Tuple[str, int], float] = {
    (i, a): 0.0 for i in I for a in A  # <-- ages 1‑4 get real carry‑over backlog
}

# Shop parameters (feel free to change)
params = dict(
    M=65,
    B_reg=2,
    B_ot=1,
    Q=1.0,
    c_reg=150.0,
    c_ot=300.0,
    gamma=50.0,
)
# ----------------------------------------------------------------------------
# ↓↓↓ END INPUT DATA ----------------------------------------------------------
# ----------------------------------------------------------------------------


def build_earlycon_model(
    I: List[str],
    T: List[int],
    A: List[int],
    d: Dict[Tuple[str, int], float],
    s_init: Dict[Tuple[str, int], float],
    params: dict,
) -> gp.Model:
    """Return a fully‑built Gurobi model for the EarlyCon scheduling MILP."""

    M       = params.get("M", 65)
    B_reg   = params.get("B_reg", 2)
    B_ot    = params.get("B_ot", 1)
    Q       = params.get("Q", 1.0)
    c_reg   = params.get("c_reg", 150.0)
    c_ot    = params.get("c_ot", 300.0)
    gamma   = params.get("gamma", 50.0)

    C_reg = M * B_reg
    C_ot  = M * B_ot

    m = gp.Model("EarlyCon")

    # ----------------------
    # Variables
    # ----------------------
    s = m.addVars(I, T, A, name="s", lb=0.0)
    p = m.addVars(I, T, A, name="p", lb=0.0)
    n_reg = m.addVars(I, T, vtype=GRB.INTEGER, name="n_reg", lb=0)
    n_ot  = m.addVars(I, T, vtype=GRB.INTEGER, name="n_ot",  lb=0)

    # ----------------------
    # Constraints
    # ----------------------
    # 1. Initial backlog for ages 1‑4 (age‑0 handled by arrivals below)
    for i in I:
        for a in A[1:]:
            m.addConstr(s[i, 0, a] == s_init[i, a], name=f"InitBacklog[{i},{a}]")
        if s_init.get((i, 0), 0.0) > 1e-8:
            print(f"⚠️  Warning: s_init[{i},0] is {s_init[(i,0)]} – move this into d[{i},0]")

    # 2. Arrivals (define age‑0 backlog for each day)
    for i in I:
        for t in T:
            m.addConstr(s[i, t, 0] == d[i, t], name=f"Arrivals[{i},{t}]")

    # 3. Backlog ageing for t ≥ 1, a ≥ 1
    for i in I:
        for t in T[1:]:
            for a in A[1:]:
                m.addConstr(
                    s[i, t, a] == s[i, t - 1, a - 1] - p[i, t - 1, a - 1],
                    name=f"Ageing[{i},{t},{a}]",
                )

    # 4. Process only what exists
    m.addConstrs((p[i, t, a] <= s[i, t, a] for i in I for t in T for a in A),
                 name="ProcExists")

    # 5. Due‑date clearance for age 4
    m.addConstrs((p[i, t, 4] == s[i, t, 4] for i in I for t in T),
                 name="DueDate")

    # 6. Batch‑volume link
    m.addConstrs((gp.quicksum(p[i, t, a] for a in A) <= (n_reg[i, t] + n_ot[i, t]) * Q
                   for i in I for t in T),
                 name="BatchVol")

    # 7. Site capacity (regular & OT)
    m.addConstrs((gp.quicksum(n_reg[i, t] for i in I) <= C_reg for t in T),
                 name="CapReg")
    m.addConstrs((gp.quicksum(n_ot[i, t] for i in I) <= C_ot for t in T),
                 name="CapOT")
    
    # 8. final day
    m.addConstrs((p[i, max(T), a] == s[i, max(T), a] for i in I for a in A[:4]),
                 name="finalDay")
    

    # ----------------------
    # Objective: maximise profit
    # ----------------------
    bonus_term = gp.quicksum(gamma * (4 - a) * p[i, t, a]
                             for i in I for t in T for a in A)
    cost_term  = gp.quicksum(c_reg * n_reg[i, t] + c_ot * n_ot[i, t]
                             for i in I for t in T)
    m.setObjective(bonus_term - cost_term, GRB.MAXIMIZE)

    # Solver output flag (1 = on, 0 = silent)
    m.Params.OutputFlag = 1

    return m

# ---------------------------------------------------------------------------
# Build & solve (example) ----------------------------------------------------
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    model = build_earlycon_model(I, T, A, d, s_init, params)
    model.optimize()

    # --- Example: print total profit
    if model.Status == GRB.OPTIMAL:
        print("Optimal objective:", model.ObjVal)


Set parameter OutputFlag to value 1
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 24.04.2 LTS")

CPU model: Intel(R) Xeon(R) Platinum 8370C CPU @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 126 rows, 108 columns and 306 nonzeros
Model fingerprint: 0xdf694c3b
Variable types: 90 continuous, 18 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e+01, 3e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+01, 1e+02]
Found heuristic solution: objective -7565.000000
Presolve removed 108 rows and 81 columns
Presolve time: 0.00s
Presolved: 18 rows, 27 columns, 60 nonzeros
Variable types: 9 continuous, 18 integer (0 binary)

Root relaxation: objective 7.200000e+03, 21 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf 

In [ ]:
# ---------------------------------------------------------------------------
# Helpers to extract and print decisions
# ---------------------------------------------------------------------------
def print_daily_report(
    model: gp.Model,
    I: List[str],
    T: List[int],
    A: List[int],
    p, n_reg, n_ot,
    params: dict,
) -> None:
    """Pretty-print the schedule and profit line by line."""

    gamma, c_reg, c_ot = params["gamma"], params["c_reg"], params["c_ot"]

    header = f"{'Day':>3} | {'Reg':>4} {'OT':>4} | {'Vol proc.':>9} | " \
             f"{'Bonus €':>9} {'Cost €':>9} {'Profit €':>9}"
    print("\n" + header)
    print("-"*len(header))

    grand_bonus = grand_cost = 0.0
    for t in T:
        day_bonus = sum(gamma*(4-a)*p[i,t,a].X for i in I for a in A)
        day_cost  = (
            c_reg * sum(n_reg[i,t].X for i in I) +
            c_ot  * sum(n_ot[i,t].X  for i in I)
        )
        day_profit = day_bonus - day_cost
        grand_bonus += day_bonus
        grand_cost  += day_cost

        print(f"{t:3d} | "
              f"{int(sum(n_reg[i,t].X for i in I)):4d} "
              f"{int(sum(n_ot[i,t].X  for i in I)):4d} | "
              f"{sum(p[i,t,a].X for i in I for a in A):9.1f} | "
              f"{day_bonus:9.0f} {day_cost:9.0f} {day_profit:9.0f}")

    print("-"*len(header))
    print(f"{'Σ':>3} | ---- ---- | --------- | "
          f"{grand_bonus:9.0f} {grand_cost:9.0f} "
          f"{grand_bonus-grand_cost:9.0f}\n")


In [ ]:
if model.Status == GRB.OPTIMAL:
    print("Optimal objective:", model.ObjVal)

    # Extract variable dictionaries from the model
    p = model.getVarByName("p")
    n_reg = model.getVarByName("n_reg")
    n_ot = model.getVarByName("n_ot")

    # If the above does not work (returns None), reconstruct using model._vars
    # But since you used m.addVars(...), you can access them as model.getVars()
    # Let's reconstruct the variable dictionaries:
    p_vars = {tuple((v.VarName.split('[')[1][:-1]).split(',')): v for v in model.getVars() if v.VarName.startswith('p[')}
    n_reg_vars = {tuple((v.VarName.split('[')[1][:-1]).split(',')): v for v in model.getVars() if v.VarName.startswith('n_reg[')}
    n_ot_vars = {tuple((v.VarName.split('[')[1][:-1]).split(',')): v for v in model.getVars() if v.VarName.startswith('n_ot[')}

    # Convert keys to correct types
    def tuple_key(key):
        return tuple(int(x) if x.isdigit() else x for x in key)

    p = {(k[0], int(k[1]), int(k[2])): v for k, v in ((tuple_key(k), v) for k, v in p_vars.items())}
    n_reg = {(k[0], int(k[1])): v for k, v in ((tuple_key(k), v) for k, v in n_reg_vars.items())}
    n_ot = {(k[0], int(k[1])): v for k, v in ((tuple_key(k), v) for k, v in n_ot_vars.items())}

    # Call the reporter
    print_daily_report(
        model, I, T, A,
        p,
        n_reg,
        n_ot,
        params,
    )

Optimal objective: 6555.0

Day |  Reg   OT | Vol proc. |   Bonus €    Cost €  Profit €
-----------------------------------------------------------
  0 |  130   41 |     170.7 |     34140     31800      2340
  1 |  130   41 |     170.7 |     34130     31800      2330
  2 |  130   43 |     171.6 |     34285     32400      1885
-----------------------------------------------------------
  Σ | ---- ---- | --------- |    102555     96000      6555



Try to optimize the solver for todays decision. 

In [ ]:
"""
Gurobi MILP model for EarlyCon (30‑day horizon, FIFO, fixed early‑delivery bonus)
Author: Lars Buchholz • 2025‑07‑14

Usage summary
-------------
1. **Edit the INPUT DATA block**
   * `I` … list of coating IDs.
   * `d[(i,t)]` … *actual* arrivals for `t = 0`, *forecast* for `t = 1‥29`.
   * `s_init[(i,a)]` … backlog carried over from the *previous* day (ages 1‑4).
2. Run the file (`python earlycon_milp.py`).  Gurobi will solve and print the
   objective.  The model is now guaranteed to be feasible for consistent data.
3. Advance one day: shift forecasts, compute new `s_init` from today’s unsolved
   backlog (ages advance by +1), set fresh `d[(i,0)]` and re‑solve.

Key parameters (can be tweaked without touching the maths):
* Machines            M          (default 65)
* Regular batches     B_reg = 2  per machine per day
* OT batches          B_ot  = 1  per machine per day
* Batch capacity      Q     = 1  (max fill‑rate)
* Cost (regular)      c_reg = €150 per batch
* Cost (overtime)     c_ot  = €300 per batch
* Early‑bonus coeff.  gamma = €50 per early day per **full** fill‑rate

All constraints implement the Markdown spec agreed with the user.
"""

from __future__ import annotations

import gurobipy as gp
from gurobipy import GRB
from typing import Dict, List, Tuple

# ----------------------------------------------------------------------------
# ↑↑↑ INPUT DATA – EDIT THIS BLOCK -------------------------------------------
# ----------------------------------------------------------------------------
# Coating IDs (can be strings or ints)
I: List[str] = ["C1", "C2", "C3"]

# Planning horizon (fixed 30 days, 0 = today)
T: List[int] = list(range(5))
A: List[int] = list(range(5))  # age buckets 0..4

# Demand forecast d[i,t]
# * t = 0  → **actual arrivals** observed this morning (after yesterday’s solve)
# * t ≥ 1 → forecast for the next 29 days
# Replace the zeros with your real data each day ↓↓↓

d: Dict[Tuple[str, int], float] = {
    **{("C1", t): 50.2 for t in T},  # <-- UPDATE HERE
    **{("C2", t): 60.1 for t in T},  # <-- UPDATE HERE
    **{("C3", t): 60.7 for t in T},  # <-- UPDATE HERE
}

# Starting backlog carried into today (ages 1‑4 only)
# Typically produced by yesterday’s solve – default 0 for demo
s_init: Dict[Tuple[str, int], float] = {
    (i, a): 0.0 for i in I for a in A  # <-- ages 1‑4 get real carry‑over backlog
}

# Shop parameters (feel free to change)
params = dict(
    M=65,
    B_reg=2,
    B_ot=1,
    Q=1.0,
    c_reg=150.0,
    c_ot=300.0,
    gamma=50.0,
)
# ----------------------------------------------------------------------------
# ↓↓↓ END INPUT DATA ----------------------------------------------------------
# ----------------------------------------------------------------------------


def build_earlycon_model(
    I: List[str],
    T: List[int],
    A: List[int],
    d: Dict[Tuple[str, int], float],
    s_init: Dict[Tuple[str, int], float],
    params: dict,
) -> gp.Model:
    """Return a fully‑built Gurobi model for the EarlyCon scheduling MILP."""

    M       = params.get("M", 65)
    B_reg   = params.get("B_reg", 2)
    B_ot    = params.get("B_ot", 1)
    Q       = params.get("Q", 1.0)
    c_reg   = params.get("c_reg", 150.0)
    c_ot    = params.get("c_ot", 300.0)
    gamma   = params.get("gamma", 50.0)

    C_reg = M * B_reg
    C_ot  = M * B_ot

    m = gp.Model("EarlyCon")

    # ----------------------
    # Variables
    # ----------------------
    s = m.addVars(I, T, A, name="s", lb=0.0)
    p = m.addVars(I, T, A, name="p", lb=0.0)
    n_reg = m.addVars(I, T, vtype=GRB.INTEGER, name="n_reg", lb=0, ub=B_reg*M)
    n_ot  = m.addVars(I, T, vtype=GRB.INTEGER, name="n_ot",  lb=0, ub=B_ot*M)

    # ----------------------
    # Constraints
    # ----------------------
    # 1. Initial backlog for ages 1‑4 (age‑0 handled by arrivals below)
    for i in I:
        for a in A[1:]:
            m.addConstr(s[i, 0, a] == s_init[i, a], name=f"InitBacklog[{i},{a}]")
        if s_init.get((i, 0), 0.0) > 1e-8:
            print(f"⚠️  Warning: s_init[{i},0] is {s_init[(i,0)]} – move this into d[{i},0]")

    # 2. Arrivals (define age‑0 backlog for each day)
    for i in I:
        for t in T:
            m.addConstr(s[i, t, 0] == d[i, t], name=f"Arrivals[{i},{t}]")

    # 3. Backlog ageing for t ≥ 1, a ≥ 1
    for i in I:
        for t in T[1:]:
            for a in A[1:]:
                m.addConstr(
                    s[i, t, a] == s[i, t - 1, a - 1] - p[i, t - 1, a - 1],
                    name=f"Ageing[{i},{t},{a}]",
                )

    # 4. Process only what exists
    m.addConstrs((p[i, t, a] <= s[i, t, a] for i in I for t in T for a in A),
                 name="ProcExists")

    # 5. Due‑date clearance for age 4
    m.addConstrs((p[i, t, 4] == s[i, t, 4] for i in I for t in T),
                 name="DueDate")

    # 6. Batch‑volume link
    m.addConstrs((gp.quicksum(p[i, t, a] for a in A) <= (n_reg[i, t] + n_ot[i, t]) * Q
                   for i in I for t in T),
                 name="BatchVol")

    # 7. Site capacity (regular & OT)
    m.addConstrs((gp.quicksum(n_reg[i, t] for i in I) <= C_reg for t in T),
                 name="CapReg")
    m.addConstrs((gp.quicksum(n_ot[i, t] for i in I) <= C_ot for t in T),
                 name="CapOT")
    
    # # 8. final day
    # m.addConstrs((p[i, max(T), a] == s[i, max(T), a] for i in I for a in A[:4]),
    #              name="finalDay")
    

    # ----------------------
    # Objective: maximise profit
    # ----------------------
    lambda_pen = 100100.0   # try values 150 … 600

    # after defining `bonus_term` and `cost_term`
    tail_penalty = lambda_pen * gp.quicksum((s[i, max(T), a]-p[i,max(T),a]) for i in I for a in A[:4])
    
    bonus_term = gp.quicksum(gamma * (4 - a) * p[i, t, a]
                             for i in I for t in T for a in A)
    cost_term  = gp.quicksum(c_reg * n_reg[i, t] + c_ot * n_ot[i, t]
                             for i in I for t in T)
    m.setObjective(bonus_term - cost_term - tail_penalty, GRB.MAXIMIZE)
    # m.setObjective(bonus_term - cost_term, GRB.MAXIMIZE)

    # Solver output flag (1 = on, 0 = silent)
    m.Params.OutputFlag = 1

    return m

# ---------------------------------------------------------------------------
# Build & solve (example) ----------------------------------------------------
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    model = build_earlycon_model(I, T, A, d, s_init, params)
    model.Params.TimeLimit =  60        # seconds
    # model.Params.MIPGap    = 0.001      # 0.1 % optimality gap
    model.optimize()

    # --- Example: print total profit
    if model.Status == GRB.OPTIMAL:
        print("Optimal objective:", model.ObjVal)


Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 60
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 24.04.2 LTS")

CPU model: Intel(R) Xeon(R) Platinum 8370C CPU @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Non-default parameters:
TimeLimit  60

Optimize a model with 190 rows, 180 columns and 486 nonzeros
Model fingerprint: 0x81a09f59
Variable types: 150 continuous, 30 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e+01, 1e+05]
  Bounds range     [6e+01, 1e+02]
  RHS range        [5e+01, 1e+02]
Found heuristic solution: objective -4.39111e+07
Presolve removed 153 rows and 105 columns
Presolve time: 0.00s
Presolved: 37 rows, 75 columns, 147 nonzeros
Variable types: 45 continuous, 30 integer (0 binary)

Root relaxation: objective 1.200000e+04, 40 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current No

In [ ]:
# ---------------------------------------------------------------------------
# Helpers to extract and print decisions
# ---------------------------------------------------------------------------
def print_daily_report(
    model: gp.Model,
    I: List[str],
    T: List[int],
    A: List[int],
    p, n_reg, n_ot,
    params: dict,
) -> None:
    """Pretty-print the schedule and profit line by line."""

    gamma, c_reg, c_ot = params["gamma"], params["c_reg"], params["c_ot"]

    header = f"{'Day':>3} | {'Reg':>4} {'OT':>4} | {'Vol proc.':>9} | " \
             f"{'Bonus €':>9} {'Cost €':>9} {'Profit €':>9}"
    print("\n" + header)
    print("-"*len(header))

    grand_bonus = grand_cost = 0.0
    for t in T:
        day_bonus = sum(gamma*(4-a)*p[i,t,a].X for i in I for a in A)
        day_cost  = (
            c_reg * sum(n_reg[i,t].X for i in I) +
            c_ot  * sum(n_ot[i,t].X  for i in I)
        )
        day_profit = day_bonus - day_cost
        grand_bonus += day_bonus
        grand_cost  += day_cost

        print(f"{t:3d} | "
              f"{int(sum(n_reg[i,t].X for i in I)):4d} "
              f"{int(sum(n_ot[i,t].X  for i in I)):4d} | "
              f"{sum(p[i,t,a].X for i in I for a in A):9.1f} | "
              f"{day_bonus:9.0f} {day_cost:9.0f} {day_profit:9.0f}")

    print("-"*len(header))
    print(f"{'Σ':>3} | ---- ---- | --------- | "
          f"{grand_bonus:9.0f} {grand_cost:9.0f} "
          f"{grand_bonus-grand_cost:9.0f}\n")


In [ ]:
# if model.Status == GRB.OPTIMAL:
print("Optimal objective:", model.ObjVal)

# Extract variable dictionaries from the model
p = model.getVarByName("p")
n_reg = model.getVarByName("n_reg")
n_ot = model.getVarByName("n_ot")

# If the above does not work (returns None), reconstruct using model._vars
# But since you used m.addVars(...), you can access them as model.getVars()
# Let's reconstruct the variable dictionaries:
p_vars = {tuple((v.VarName.split('[')[1][:-1]).split(',')): v for v in model.getVars() if v.VarName.startswith('p[')}
n_reg_vars = {tuple((v.VarName.split('[')[1][:-1]).split(',')): v for v in model.getVars() if v.VarName.startswith('n_reg[')}
n_ot_vars = {tuple((v.VarName.split('[')[1][:-1]).split(',')): v for v in model.getVars() if v.VarName.startswith('n_ot[')}

# Convert keys to correct types
def tuple_key(key):
    return tuple(int(x) if x.isdigit() else x for x in key)

p = {(k[0], int(k[1]), int(k[2])): v for k, v in ((tuple_key(k), v) for k, v in p_vars.items())}
n_reg = {(k[0], int(k[1])): v for k, v in ((tuple_key(k), v) for k, v in n_reg_vars.items())}
n_ot = {(k[0], int(k[1])): v for k, v in ((tuple_key(k), v) for k, v in n_ot_vars.items())}

# Call the reporter
print_daily_report(
    model, I, T, A,
    p,
    n_reg,
    n_ot,
    params,
)

Optimal objective: 11490.000513501465

Day |  Reg   OT | Vol proc. |   Bonus €    Cost €  Profit €
-----------------------------------------------------------
  0 |  130   40 |     170.0 |     34000     31500      2500
  1 |  130   41 |     171.0 |     34160     31800      2360
  2 |  130   41 |     171.0 |     34170     31800      2370
  3 |  130   41 |     170.8 |     34130     31800      2330
  4 |  130   43 |     172.2 |     34330     32400      1930
-----------------------------------------------------------
  Σ | ---- ---- | --------- |    170790    159300     11490



Trying to combine everything for actual evaluation.

**How to run / adapt the EarlyCon MILP (quick guide)**

1. **Open the file**

2. **Edit the *INPUT-DATA* block at the end of the file**

   * **`I`** list of coating IDs (strings or ints).
   * **`total_periods`** and the three phase lengths (`warmup_periods`, `evaluation_periods`, `cooldown_periods`) must sum to the same number.
   * **`d_actual`** array of realised arrivals – size = `(len(I), total_periods)`.
   * **`d_predicted`** same shape, holding day-ahead forecasts.
   * **`dummy_forecast`** fallback value when no forecast is available.
   * **Shop parameters** (`M`, `B_reg`, …) can be tweaked inline.

3. **Run**

   The script loops day-by-day:

   * builds the 5-day MILP,
   * solves with a 60 s time-limit,
   * prints a day report,
   * rolls backlog forward.

4. **Typical adaptations**

   | What you want                       | Where to change                                                                             |
   | ----------------------------------- | ------------------------------------------------------------------------------------------- |
   | **Longer look-ahead** (e.g. 7 days) | `planning_horizon = 7` and regenerate `T = range(7)`                                        |
   | **Multi-day forecast**              | after `demand = …` insert additional forecast slots (`t = 2,3…`) before the dummy-fill loop |
   | **Different penalty**               | adjust `lambda_pen` inside `build_oerlikon_model`                                           |
   | **Speed vs. optimality**            | lower `model.Params.TimeLimit` (e.g. 10 s) or add `model.Params.MIPGap = 0.001`             |

5. **Outputs**

   * Console shows per-day batches, volume, cost, bonus, profit.
   * `decisions_df` (printed at the end) lists the processed volume per coating per calendar day; copy to Excel if needed.


In [2]:
from __future__ import annotations

import gurobipy as gp
from gurobipy import GRB
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd

def build_oerlikon_model(
    I: List[str],
    T: List[int],
    A: List[int],
    d: Dict[Tuple[str, int], float],
    s_init: Dict[Tuple[str, int], float],
    params: dict,
) -> gp.Model:
    """Return a fully‑built Gurobi model for the EarlyCon scheduling MILP."""

    M       = params.get("M", 65)
    B_reg   = params.get("B_reg", 2)
    B_ot    = params.get("B_ot", 1)
    Q       = params.get("Q", 1.0)
    c_reg   = params.get("c_reg", 150.0)
    c_ot    = params.get("c_ot", 300.0)
    gamma   = params.get("gamma", 50.0)

    C_reg = M * B_reg
    C_ot  = M * B_ot

    m = gp.Model("EarlyCon")

    # ----------------------
    # Variables
    # ----------------------
    s = m.addVars(I, T, A, name="s", lb=0.0)
    p = m.addVars(I, T, A, name="p", lb=0.0)
    n_reg = m.addVars(I, T, vtype=GRB.INTEGER, name="n_reg", lb=0, ub=B_reg*M)
    n_ot  = m.addVars(I, T, vtype=GRB.INTEGER, name="n_ot",  lb=0, ub=B_ot*M)

    # ----------------------
    # Constraints
    # ----------------------
    # 1. Initial backlog for ages 1‑4 (age‑0 handled by arrivals below)
    for i in I:
        for a in A[1:]:
            m.addConstr(s[i, 0, a] == s_init[i, a], name=f"InitBacklog[{i},{a}]")
        if s_init.get((i, 0), 0.0) > 1e-8:
            print(f"⚠️  Warning: s_init[{i},0] is {s_init[(i,0)]} – move this into d[{i},0]")

    # 2. Arrivals (define age‑0 backlog for each day)
    for i in I:
        for t in T:
            m.addConstr(s[i, t, 0] == d[i, t], name=f"Arrivals[{i},{t}]")

    # 3. Backlog ageing for t ≥ 1, a ≥ 1
    for i in I:
        for t in T[1:]:
            for a in A[1:]:
                m.addConstr(
                    s[i, t, a] == s[i, t - 1, a - 1] - p[i, t - 1, a - 1],
                    name=f"Ageing[{i},{t},{a}]",
                )

    # 4. Process only what exists
    m.addConstrs((p[i, t, a] <= s[i, t, a] for i in I for t in T for a in A),
                 name="ProcExists")

    # 5. Due‑date clearance for age 4
    m.addConstrs((p[i, t, 4] == s[i, t, 4] for i in I for t in T),
                 name="DueDate")

    # 6. Batch‑volume link
    m.addConstrs((gp.quicksum(p[i, t, a] for a in A) <= (n_reg[i, t] + n_ot[i, t]) * Q
                   for i in I for t in T),
                 name="BatchVol")

    # 7. Site capacity (regular & OT)
    m.addConstrs((gp.quicksum(n_reg[i, t] for i in I) <= C_reg for t in T),
                 name="CapReg")
    m.addConstrs((gp.quicksum(n_ot[i, t] for i in I) <= C_ot for t in T),
                 name="CapOT")
    
    

    # ----------------------
    # Objective: maximise profit
    # ----------------------
    lambda_pen = 100100.0   # try values 150 … 600

    # after defining `bonus_term` and `cost_term`
    tail_penalty = lambda_pen * gp.quicksum((s[i, max(T), a]-p[i,max(T),a]) for i in I for a in A[:4])
    
    bonus_term = gp.quicksum(gamma * (4 - a) * p[i, t, a]
                             for i in I for t in T for a in A)
    cost_term  = gp.quicksum(c_reg * n_reg[i, t] + c_ot * n_ot[i, t]
                             for i in I for t in T)
    m.setObjective(bonus_term - cost_term - tail_penalty, GRB.MAXIMIZE)

    # Solver output flag (1 = on, 0 = silent)
    m.Params.OutputFlag = 1

    return m

def documentation(model: gp.Model, I, T, A, params) -> None:
    """Output the documentation string for the EarlyCon scheduling MILP."""
    
    if model.Status == GRB.OPTIMAL:
        print("Optimal objective:", model.ObjVal)
    else:
        print("Model not solved to optimality. Status:", model.Status)
        print("Currently, the objective value is:", model.ObjVal if model.ObjVal is not None else "N/A")

    # Extract variable dictionaries from the model
    p = model.getVarByName("p")
    n_reg = model.getVarByName("n_reg")
    n_ot = model.getVarByName("n_ot")

    # If the above does not work (returns None), reconstruct using model._vars
    # But since you used m.addVars(...), you can access them as model.getVars()
    # Let's reconstruct the variable dictionaries:
    p_vars = {tuple((v.VarName.split('[')[1][:-1]).split(',')): v for v in model.getVars() if v.VarName.startswith('p[')}
    n_reg_vars = {tuple((v.VarName.split('[')[1][:-1]).split(',')): v for v in model.getVars() if v.VarName.startswith('n_reg[')}
    n_ot_vars = {tuple((v.VarName.split('[')[1][:-1]).split(',')): v for v in model.getVars() if v.VarName.startswith('n_ot[')}

    # Convert keys to correct types
    def tuple_key(key):
        return tuple(int(x) if x.isdigit() else x for x in key)

    p = {(k[0], int(k[1]), int(k[2])): v for k, v in ((tuple_key(k), v) for k, v in p_vars.items())}
    n_reg = {(k[0], int(k[1])): v for k, v in ((tuple_key(k), v) for k, v in n_reg_vars.items())}
    n_ot = {(k[0], int(k[1])): v for k, v in ((tuple_key(k), v) for k, v in n_ot_vars.items())}

    # Call the reporter
    print_daily_report(
        model, I, T, A,
        p,
        n_reg,
        n_ot,
        params,
    )
    
# Helpers to extract and print decisions
def print_daily_report(
    model: gp.Model,
    I: List[str],
    T: List[int],
    A: List[int],
    p, n_reg, n_ot,
    params: dict,
) -> None:
    """Pretty-print the schedule and profit line by line."""

    gamma, c_reg, c_ot = params["gamma"], params["c_reg"], params["c_ot"]

    header = f"{'Day':>3} | {'Reg':>4} {'OT':>4} | {'Vol proc.':>9} | " \
             f"{'Bonus €':>9} {'Cost €':>9} {'Profit €':>9}"
    print("\n" + header)
    print("-"*len(header))

    grand_bonus = grand_cost = 0.0
    for t in T:
        day_bonus = sum(gamma*(4-a)*p[i,t,a].X for i in I for a in A)
        day_cost  = (
            c_reg * sum(n_reg[i,t].X for i in I) +
            c_ot  * sum(n_ot[i,t].X  for i in I)
        )
        day_profit = day_bonus - day_cost
        grand_bonus += day_bonus
        grand_cost  += day_cost

        print(f"{t:3d} | "
              f"{int(sum(n_reg[i,t].X for i in I)):4d} "
              f"{int(sum(n_ot[i,t].X  for i in I)):4d} | "
              f"{sum(p[i,t,a].X for i in I for a in A):9.1f} | "
              f"{day_bonus:9.0f} {day_cost:9.0f} {day_profit:9.0f}")

    print("-"*len(header))
    print(f"{'Σ':>3} | ---- ---- | --------- | "
          f"{grand_bonus:9.0f} {grand_cost:9.0f} "
          f"{grand_bonus-grand_cost:9.0f}\n")

    


if __name__ == "__main__":
    
    # --- INPUT DATA ----------------------------------------------------------
    total_periods= 13
    warmup_periods = 5
    evaluation_periods = 8
    cooldown_periods = 0
    
    # Planning horizon (how many forecast day do we consider)
    planning_horizon = 5
    
    # Coating IDs (can be strings or ints)
    I: List[str] = ["C1", "C12", "C15"]
    
    # Planning horizon (fixed 30 days, 0 = today)
    T: List[int] = list(range(planning_horizon)) 
    A: List[int] = list(range(5))  # age buckets 0..4  
    
    # Shop parameters
    params = dict(
        M=3,
        B_reg=2,
        B_ot=1,
        Q=1.0,
        c_reg=150.0,
        c_ot=300.0,
        gamma=50.0,
    )
    
    # Demand forecast d[i,t]
    
    d_predicted: Dict[Tuple[str, int], float] = {
        

        # Forecasts for t=1 to t=7 (assuming T is list(range(8)))
        # You would fill this with your forecast data for each day and coating
        ("C1", 1): 0.664382,
        ("C1", 2): 3.116381,
        ("C1", 3): 2.638918,
        ("C1", 4): 0.006072,
        ("C1", 5): 0.16748,
        ("C1", 6): 7.168205,
        ("C1", 7): 2.797644,
        ("C1", 8): 2.601825,
        ("C1", 9): 2.337529,
        ("C1", 10): 2.595371,
        ("C1", 11): 0.003316,
        ("C1", 12): 0.037442,
        ("C1", 13): 11.13356,

        ("C12", 1): 0.092957,
        ("C12", 2): 1.709557,
        ("C12", 3): 1.02712,
        ("C12", 4): 0.17251,
        ("C12", 5): 0.014203,
        ("C12", 6): 5.333041,
        ("C12", 7): 1.118007,
        ("C12", 8): 1.141394,
        ("C12", 9): 0.764299,
        ("C12", 10): 1.316963,
        ("C12", 11): 0.24267,
        ("C12", 12): 0.11134,
        ("C12", 13): 11.81268,

        ("C15", 1): 0.420059,
        ("C15", 2): 1.890447,
        ("C15", 3): 1.505981,
        ("C15", 4): 0.054836,
        ("C15", 5): 0.169436,
        ("C15", 6): 4.827995,
        ("C15", 7): 1.741779,
        ("C15", 8): 1.505855,
        ("C15", 9): 1.322614,
        ("C15", 10): 1.535442,
        ("C15", 11): 0.027544,
        ("C15", 12): 0.007686,
        ("C15", 13): 6.440917,
    }
    
    # Actual demand
    d_actual: Dict[Tuple[str, int], float] = {
        
    
        ("C1", 0): 3.2,
        ("C1", 1): 0.784,
        ("C1", 2): 2.919,
        ("C1", 3): 2.341,
        ("C1", 4): 0.092,
        ("C1", 5): 0.309,
        ("C1", 6): 6.547,
        ("C1", 7): 2.932,
        ("C1", 8): 3.119,
        ("C1", 9): 2.067,
        ("C1", 10): 1.774,
        ("C1", 11): 0.027,
        ("C1", 12): 0.154,
        ("C1", 13): 10.438,

        ("C12", 0): 6.4,
        ("C12", 1): 0.136,
        ("C12", 2): 1.302,
        ("C12", 3): 0.964,
        ("C12", 4): 0.289,
        ("C12", 5): 0.055,
        ("C12", 6): 6.229,
        ("C12", 7): 1.624,
        ("C12", 8): 1.293,
        ("C12", 9): 0.833,
        ("C12", 10): 1.471,
        ("C12", 11): 0.196,
        ("C12", 12): 0.082,
        ("C12", 13): 12.489,

        ("C15", 0): 2.5,
        ("C15", 1): 0.375,
        ("C15", 2): 2.131,
        ("C15", 3): 1.229,
        ("C15", 4): 0.077,
        ("C15", 5): 0.288,
        ("C15", 6): 5.116,
        ("C15", 7): 1.503,
        ("C15", 8): 1.412,
        ("C15", 9): 1.176,
        ("C15", 10): 1.644,
        ("C15", 11): 0.045,
        ("C15", 12): 0.019,
        ("C15", 13): 6.921,
    }
    
    dummy_forecast = 5  # <-- Dummy value for all periods where no forecast is available
    
    # Starting backlog carried into today (ages 1‑4 only)
    # Updated by yesterday’s solve – initially 0
    s_init: Dict[Tuple[str, int], float] = {
        (i, a): 0.0 for i in I for a in A  # <-- ages 1‑4 get real carry‑over backlog
    }
    
    # End of INPUT DATA ----------------------------------------------------------

    print(f"Starting Oerlikon model with a planning horizon of {planning_horizon} days.")
    
    print(f"Total periods: {total_periods}, "
              f"Warmup: {warmup_periods}, "
              f"Evaluation: {evaluation_periods}, "
              f"Cooldown: {cooldown_periods}")
    
    if total_periods-warmup_periods-evaluation_periods-cooldown_periods != 0:
        raise ValueError("Total periods must equal warmup + evaluation + cooldown periods")
    else:   
        print("Total periods match warmup, evaluation, and cooldown periods.")
        
    print("Starting warmup period:")
    total_period= 0
    total_costs = 0.0
    total_profit = 0.0
    c_reg   = params.get("c_reg", 150.0)
    c_ot    = params.get("c_ot", 300.0)
    gamma   = params.get("gamma", 50.0)
    
    decisions= np.zeros((total_periods, len(I)), dtype=float)
    
    # warmup_periods
    for w in range(warmup_periods):
        total_period += 1
        print(f"Warmup period {w+1}/{warmup_periods}")
        
        
        # Update the demand fed to the model
        if total_period != total_periods:
            demand: Dict[Tuple[str, int], float] = {
                **{(i, t): d_actual[(i,total_period-1)] for i in I for t in T[:1]},  # actual for today
                **{(i, t): d_predicted[(i,total_period)] for i in I for t in T[1:2]},  # predicted for next day
                **{(i, t): dummy_forecast for t in T if t not in [0, 1] for i in I}, # dummy for future
            }
        else:
            demand = {
                **{(i, t): d_actual[(i,total_period-1)] for i in I for t in T[:1]},  # actual for today
                **{(i, t): dummy_forecast for t in T[1:] for i in I},  # dummy for future
            }
        
        print(f"Demand for this run: {demand}")
        print(f"Initial backlog for this run: {s_init}")
        
        # apply model
        model = build_oerlikon_model(I, T, A, demand, s_init, params)
        model.Params.TimeLimit =  60        # seconds
        model.optimize()
        documentation(model=model, I=I, T=T, A=A, params=params)
        
        # Extract variables from the model        
        # one helper you can re-use 
        def grab_td(name, keys): 
            return {k: model.getVarByName(f"{name}[{','.join(map(str,k))}]") for k in keys} 
        
        p = grab_td("p" , [(i,t,a) for i in I for t in T for a in A]) 
        n_reg = grab_td("n_reg", [(i,t) for i in I for t in T]) 
        n_ot = grab_td("n_ot" , [(i,t) for i in I for t in T]) 
        s_var = grab_td("s" , [(i,t,a) for i in I for t in T for a in A])
        
        decisions[total_period-1, :] = [p[i, 0, 0].X for i in I]  # Store decisions for today
        
        # Update tomorrow's s_init
        for i in I:
            for a in A[1:]:
                s_init[i,a] = s_var[(i,1,a)].X
        
        # Calculate the costs occured today
        daily_costs = sum(c_reg * n_reg[(i,0)].X + c_ot * n_ot [(i,0)].X for i in I)
        print(f"Number of regular batches today: {sum(n_reg[(i,0)].X for i in I)}")
        print(f"Number of overtime batches today: {sum(n_ot[(i,0)].X for i in I)}")
        print(f"Daily costs for today: {daily_costs}")    
        
        # Calculate the profit (early customer delivery) today
        daily_profit = sum(gamma * (4 - a) * p[i, 0, a].X for i in I for a in A)
        print(f"Number of products processed today: {sum(p[i, 0, a].X for i in I for a in A)}")
        print(f"Daily profit for today: {daily_profit}")
    
    for w in range(evaluation_periods):
        total_period += 1
        print(f"Evaluation period {w+1}/{evaluation_periods}")
        
        
        # Update the demand fed to the model
        if total_period != total_periods:
            demand: Dict[Tuple[str, int], float] = {
                **{(i, t): d_actual[(i,total_period-1)] for i in I for t in T[:1]},  # actual for today
                **{(i, t): d_predicted[(i,total_period)] for i in I for t in T[1:2]},  # predicted for next day
                **{(i, t): dummy_forecast for t in T if t not in [0, 1] for i in I}, # dummy for future
            }
        else:
            demand = {
                **{(i, t): d_actual[(i,total_period-1)] for i in I for t in T[:1]},  # actual for today
                **{(i, t): dummy_forecast for t in T[1:] for i in I},  # dummy for future
            }
        
        print(f"Demand for this run: {demand}")
        print(f"Initial backlog for this run: {s_init}")
        
        # apply model
        model = build_oerlikon_model(I, T, A, demand, s_init, params)
        model.Params.TimeLimit =  60        # seconds
        model.optimize()
        documentation(model=model, I=I, T=T, A=A, params=params)
        
        # Extract variables from the model        
        # one helper you can re-use 
        def grab_td(name, keys): 
            return {k: model.getVarByName(f"{name}[{','.join(map(str,k))}]") for k in keys} 
        
        p = grab_td("p" , [(i,t,a) for i in I for t in T for a in A]) 
        n_reg = grab_td("n_reg", [(i,t) for i in I for t in T]) 
        n_ot = grab_td("n_ot" , [(i,t) for i in I for t in T]) 
        s_var = grab_td("s" , [(i,t,a) for i in I for t in T for a in A])
        
        decisions[total_period-1, :] = [p[i, 0, 0].X for i in I]  # Store decisions for today
        
        # Update tomorrow's s_init
        for i in I:
            for a in A[1:]:
                s_init[i,a] = s_var[(i,1,a)].X
        
        # Calculate the costs occured today
        daily_costs = sum(c_reg * n_reg[(i,0)].X + c_ot * n_ot [(i,0)].X for i in I)
        print(f"Number of regular batches today: {sum(n_reg[(i,0)].X for i in I)}")
        print(f"Number of overtime batches today: {sum(n_ot[(i,0)].X for i in I)}")
        print(f"Daily costs for today: {daily_costs}")
        total_costs += daily_costs
        print(f"Total costs so far: {total_costs}")   
        
        # Calculate the profit (early customer delivery) today
        daily_profit = sum(gamma * (4 - a) * p[i, 0, a].X for i in I for a in A)
        print(f"Number of products processed today: {sum(p[i, 0, a].X for i in I for a in A)}")
        print(f"Daily profit for today: {daily_profit}")
        total_profit += daily_profit
        print(f"Total profit so far: {total_profit}")
    
    for w in range(cooldown_periods):
        total_period += 1
        print(f"Cooldown period {w+1}/{cooldown_periods}")
        
        
        # Update the demand fed to the model
        if total_period != total_periods:
            demand: Dict[Tuple[str, int], float] = {
                **{(i, t): d_actual[(i,total_period-1)] for i in I for t in T[:1]},  # actual for today
                **{(i, t): d_predicted[(i,total_period)] for i in I for t in T[1:2]},  # predicted for next day
                **{(i, t): dummy_forecast for t in T if t not in [0, 1] for i in I}, # dummy for future
            }
        else:
            demand = {
                **{(i, t): d_actual[(i,total_period-1)] for i in I for t in T[:1]},  # actual for today
                **{(i, t): dummy_forecast for t in T[1:] for i in I},  # dummy for future
            }
        
        print(f"Demand for this run: {demand}")
        print(f"Initial backlog for this run: {s_init}")
        
        # apply model
        model = build_oerlikon_model(I, T, A, demand, s_init, params)
        model.Params.TimeLimit =  60        # seconds
        model.optimize()
        documentation(model=model, I=I, T=T, A=A, params=params)
        
        # Extract variables from the model        
        # one helper you can re-use 
        def grab_td(name, keys): 
            return {k: model.getVarByName(f"{name}[{','.join(map(str,k))}]") for k in keys} 
        
        p = grab_td("p" , [(i,t,a) for i in I for t in T for a in A]) 
        n_reg = grab_td("n_reg", [(i,t) for i in I for t in T]) 
        n_ot = grab_td("n_ot" , [(i,t) for i in I for t in T]) 
        s_var = grab_td("s" , [(i,t,a) for i in I for t in T for a in A])
        
        decisions[total_period-1, :] = [p[i, 0, 0].X for i in I]  # Store decisions for today
        
        # Update tomorrow's s_init
        for i in I:
            for a in A[1:]:
                s_init[i,a] = s_var[(i,1,a)].X
        
        # Calculate the costs occured today
        daily_costs = sum(c_reg * n_reg[(i,0)].X + c_ot * n_ot [(i,0)].X for i in I)
        print(f"Number of regular batches today: {sum(n_reg[(i,0)].X for i in I)}")
        print(f"Number of overtime batches today: {sum(n_ot[(i,0)].X for i in I)}")
        print(f"Daily costs for today: {daily_costs}")    
        
        # Calculate the profit (early customer delivery) today
        daily_profit = sum(gamma * (4 - a) * p[i, 0, a].X for i in I for a in A)
        print(f"Number of products processed today: {sum(p[i, 0, a].X for i in I for a in A)}")
        print(f"Daily profit for today: {daily_profit}")
    
    print("Finished all periods.")
    print(f"FInal total costs: {total_costs}")
    print(f"FInal total profit: {total_profit}")
    
    decisions_df = pd.DataFrame(decisions, columns=I)
    print("Decisions table:")
    display(decisions_df)
    
    
        
        

Starting Oerlikon model with a planning horizon of 5 days.
Total periods: 13, Warmup: 5, Evaluation: 8, Cooldown: 0
Total periods match warmup, evaluation, and cooldown periods.
Starting warmup period:
Warmup period 1/5
Demand for this run: {('C1', 0): 3.2, ('C12', 0): 6.4, ('C15', 0): 2.5, ('C1', 1): 0.664382, ('C12', 1): 0.092957, ('C15', 1): 0.420059, ('C1', 2): 5, ('C12', 2): 5, ('C15', 2): 5, ('C1', 3): 5, ('C12', 3): 5, ('C15', 3): 5, ('C1', 4): 5, ('C12', 4): 5, ('C15', 4): 5}
Initial backlog for this run: {('C1', 0): 0.0, ('C1', 1): 0.0, ('C1', 2): 0.0, ('C1', 3): 0.0, ('C1', 4): 0.0, ('C12', 0): 0.0, ('C12', 1): 0.0, ('C12', 2): 0.0, ('C12', 3): 0.0, ('C12', 4): 0.0, ('C15', 0): 0.0, ('C15', 1): 0.0, ('C15', 2): 0.0, ('C15', 3): 0.0, ('C15', 4): 0.0}
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 60
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (linux64 - "Ubuntu 24.04.2 LTS")

CPU model: Intel(R) Xeon(R) Platinum 8370C CPU @ 2.80GHz, instruction set [

,C1,C12,C15
0,3.000000,5.00000,0.000000
1,0.784000,0.13600,0.375000
2,2.919000,1.00000,2.000000
3,2.000000,0.96400,1.000000
4,0.000000,0.28900,0.000000
5,0.309000,0.05500,0.288000
6,6.000000,3.00000,0.000000
7,2.932000,1.62400,1.503000
8,3.000000,0.14700,0.381000
9,2.067000,0.83300,0.969000
